<a href="https://colab.research.google.com/github/joserlandero/DatosMasivos/blob/main/Tarea_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea 2 - RDD en Spark_Caso de uso

Para que no se nos pase un RDD por sus siglas es: Resilient Distributed Dataset la cual es escencial para distribuir las operaciones en diferentes clusters

## Dataframe
El dataframe a usar será el mismo de la clase anterior el cual es un conjunto de datos histórios de siniestros de automoviles

In [3]:
# Dataset
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tarea2") \
    .master("local[*]") \
    .getOrCreate()
sc = spark.sparkContext #RDD

rdd_text = sc.textFile("/content/Austin_Crash_Report_Data_-_Crash_Level_Records.csv") #leemos la base de datos

In [4]:
# Imprimimos el schema
rdd_text.take(1)

['ID,Crash ID,crash_fatal_fl,case_id,rpt_block_num,rpt_street_name,rpt_street_sfx,crash_speed_limit,road_constr_zone_fl,latitude,longitude,crash_sev_id,sus_serious_injry_cnt,nonincap_injry_cnt,poss_injry_cnt,non_injry_cnt,unkn_injry_cnt,tot_injry_cnt,death_cnt,units_involved,contro,point,motor_vehicle_death_count,motor_vehicle_serious_injury_count,bicycle_death_count,bicycle_serious_injury_count,pedestrian_death_count,pedestrian_serious_injury_count,motorcycle_death_count,motorcycle_serious_injury_count,other_death_count,other_serious_injury_count,onsys_fl,private_dr_fl,micromobility_serious_injury_count,micromobility_death_count,Crash timestamp (US/Central),Crash timestamp,Is deleted,Is temporary record,Law enforcement fatality count,Reported street prefix,Estimated Maximum Comprehensive Cost,Estimated Total Comprehensive Cost,Location ID,Location group,Address,Collision type']

In [5]:
# Ahora podemos hacer operaciones aritméticas
# por ejemplo obtenemos el promedio de mis siniestros
header = rdd_text.first()
data_rdd = rdd_text.filter(lambda line: line != header)

mean_amt_rdd = data_rdd.map(lambda line: line.split(',')) \
                       .filter(lambda fields: len(fields) > 42 and fields[42].strip() != '') \
                       .map(lambda fields: float(fields[42].strip()))

# media
mean = mean_amt_rdd.mean()

In [6]:
print(f"Siniestro mas alto: {mean_amt_rdd.max()}")
print(f"Siniestro mas bajo: {mean_amt_rdd.min()}")

Siniestro mas alto: 3500000.0
Siniestro mas bajo: 20000.0


In [10]:
# Tambien otra operación básica que se puede hacer es el total de registros del dataframe
print(f"Total de registros :{mean_amt_rdd.count()}")

Total de registros :125980


In [11]:
from pyspark.sql.functions import col

In [16]:
# 1. Creamos una función para filtrar de forma segura
def filter_high_cost(line):
    fields = line.split(',')
    # Verificamos que la fila tenga suficientes columnas y el campo no esté vacío
    if len(fields) > 42 and fields[42].strip() != '':
        try:
            # Convertimos a float y verificamos si es mayor a 100,000
            cost = float(fields[42].strip())
            return cost > 1000000
        except ValueError:
            # Si el valor no se puede convertir a float (ej. basura en los datos), lo descartamos
            return False
    return False

# 2. Aplicamos el filtro a nuestro RDD
filtered_rdd = data_rdd.filter(filter_high_cost)

# 3. (Opcional) Verificamos cuántos registros cumplen la condición
print(f"Registros mayores a 1,000,000: {filtered_rdd.count()}")

Registros mayores a 1,000,000: 4406
